# PySpark y Spark SQL

PySpark es la interfaz de Python de Apache Spark. En esta práctica se lee la serie de COVID-19 y se consulta con DataFrames y con SQL.

Un filtro, una agrupación o un orden se escriben en una frase SQL. La referencia de la API está en http://spark.apache.org/docs/latest/api/python/


# 1. Sesión de Spark


In [1]:
%pip install pyspark

from pyspark.sql import SparkSession
import pyspark.sql.functions as F


spark = (
    SparkSession.builder
    .appName("6.2 Spark y SQL")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

sc = spark.sparkContext
spark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\Agustin Wencelblat\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


# 2. Lectura de datos

La serie es la de Our World in Data sobre COVID-19. El archivo `owid-covid-data.csv` ya está en la carpeta de la notebook. Cada fila es un país, o un agregado, en una fecha.


In [2]:
# import requests
# path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
# req = requests.get(path)
# url_content = req.content
#
# csv_file_name = 'owid-covid-data.csv'
# csv_file = open(csv_file_name, 'wb')
#
# csv_file.write(url_content)
# csv_file.close()

df = spark.read.csv("owid-covid-data.csv", header=True, inferSchema=True)

# 3. DataFrames


In [3]:
# Nombre y tipo de cada columna
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [4]:
# date llega como texto. Esta consulta la muestra como fecha y no modifica df.
df.select(F.to_date(df.date).alias('date'))

DataFrame[date: date]

In [5]:
# Resumen de las columnas numéricas
df.describe().show()

+-------+--------+-------------+-----------+-------------------+-----------------+------------------+------------------+------------------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+------------------+-----------------+------------------------+------------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-------------------+------------------+------------------------+----------------------+------------------+-------------------------------+-------------------+------------------+-------------+--------------------+--------------------+-----------------------+--------------------+-----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-----------------

In [6]:
# Argentina, de la fecha más reciente a la más antigua
df.filter(df.location == "Argentina").orderBy(F.desc("date")).show()

+--------+-------------+---------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-----------------------------

In [7]:
# Suma de new_cases por location
df.groupBy("location").sum("new_cases").orderBy(F.desc("sum(new_cases)")).show(truncate=False)

+-----------------------------+--------------+
|location                     |sum(new_cases)|
+-----------------------------+--------------+
|World                        |775935057     |
|High-income countries        |429044052     |
|Asia                         |301564180     |
|Europe                       |252916868     |
|Upper-middle-income countries|251756125     |
|European Union (27)          |185822587     |
|North America                |124492698     |
|United States                |103436829     |
|China                        |99373219      |
|Lower-middle-income countries|92019711      |
|South America                |68811012      |
|India                        |45041748      |
|France                       |38997490      |
|Germany                      |38437756      |
|Brazil                       |37511921      |
|South Korea                  |34571873      |
|Japan                        |33803572      |
|Italy                        |26781078      |
|United Kingd

# 4. Spark SQL

`createOrReplaceTempView` publica el DataFrame como una vista de esta sesión. A partir de ahí la consulta se escribe en SQL.


In [8]:
# Vista temporal de esta sesión
df.createOrReplaceTempView("covid_data")
# Para guardarla en el catálogo:
# df.write.mode("overwrite").saveAsTable("covid_data")


In [9]:
df2 = spark.sql("SELECT * from covid_data")
df2.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [10]:
df2.show()

+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-------------------------------

In [11]:
# Cantidad de filas por location
groupDF = spark.sql("SELECT location, count(*) from covid_data group by location")
groupDF.show()


+-------------------+--------+
|           location|count(1)|
+-------------------+--------+
|           Anguilla|    1674|
|        Afghanistan|    1674|
|             Africa|    1674|
|            Algeria|    1674|
|          Argentina|    1678|
|             Angola|    1674|
|            Belgium|    1674|
|            Albania|    1674|
|            Bahamas|    1674|
|            Belarus|    1674|
|     American Samoa|    1674|
|            Andorra|    1674|
|         Bangladesh|    1674|
|           Barbados|    1674|
|              Aruba|    1674|
|         Azerbaijan|    1674|
|            Armenia|    1674|
|Antigua and Barbuda|    1674|
|               Asia|    1684|
|          Australia|    1674|
+-------------------+--------+
only showing top 20 rows


# Guia PySpark + SQL

### Ejercicio H. Total de casos confirmados por país hasta la fecha más reciente


In [13]:
query = """
SELECT location AS pais, SUM(total_cases) AS total_casos_confirmados
FROM covid_data
GROUP BY location
ORDER BY total_casos_confirmados DESC
LIMIT 10
"""

result = spark.sql(query)
result.show()

+--------------------+-----------------------+
|                pais|total_casos_confirmados|
+--------------------+-----------------------+
|               World|           715697182101|
|High-income count...|           393562796886|
|                Asia|           252167317226|
|              Europe|           236756684151|
|Upper-middle-inco...|           218037426784|
| European Union (27)|           171330559623|
|       North America|           127073670231|
|       United States|           105914483457|
|Lower-middle-inco...|           100873065026|
|       South America|            73484570403|
+--------------------+-----------------------+



### Ejercicio I. Número de muertes el 2022-01-01


In [14]:
query = """
SELECT SUM(total_deaths) AS muertes_acumuladas, SUM(new_deaths) AS muertes_nuevas_ese_dia
FROM covid_data
WHERE date = '2022-01-01' AND continent IS NOT NULL
"""

result = spark.sql(query)
result.show()

+------------------+----------------------+
|muertes_acumuladas|muertes_nuevas_ese_dia|
+------------------+----------------------+
|           5446878|                     0|
+------------------+----------------------+



### Ejercicio J: Evolucion diaria de casos en Argentina

In [15]:
query = """
SELECT date, new_cases, total_cases
FROM covid_data
WHERE location = 'Argentina'
ORDER BY date
"""

result = spark.sql(query)
result.show()

+----------+---------+-----------+
|      date|new_cases|total_cases|
+----------+---------+-----------+
|2020-01-01|     NULL|       NULL|
|2020-01-02|     NULL|       NULL|
|2020-01-03|     NULL|       NULL|
|2020-01-04|     NULL|       NULL|
|2020-01-05|        0|          0|
|2020-01-06|        0|          0|
|2020-01-07|        0|          0|
|2020-01-08|        0|          0|
|2020-01-09|        0|          0|
|2020-01-10|        0|          0|
|2020-01-11|        0|          0|
|2020-01-12|        0|          0|
|2020-01-13|        0|          0|
|2020-01-14|        0|          0|
|2020-01-15|        0|          0|
|2020-01-16|        0|          0|
|2020-01-17|        0|          0|
|2020-01-18|        0|          0|
|2020-01-19|        0|          0|
|2020-01-20|        0|          0|
+----------+---------+-----------+
only showing top 20 rows


### Ejercicio K: Calcular porcentaje de la población vacunada por país.

In [16]:
query = """
SELECT location AS pais,
    ROUND(MAX(people_vaccinated) * 100 / MAX(population), 2) AS pct_al_menos_una_dosis,
    ROUND(max(people_fully_vaccinated) * 100 / MAX(population), 2) AS pct_esquema_completo
FROM covid_data
WHERE continent IS NOT NULL
GROUP BY location
HAVING MAX(people_vaccinated) IS NOT NULL
ORDER BY pct_al_menos_una_dosis DESC
"""

result = spark.sql(query)
result.show()

+--------------------+----------------------+--------------------+
|                pais|pct_al_menos_una_dosis|pct_esquema_completo|
+--------------------+----------------------+--------------------+
|           Gibraltar|                129.07|              126.89|
|             Tokelau|                116.38|              116.38|
|               Qatar|                105.83|              105.83|
|United Arab Emirates|                105.83|              103.72|
|               Nauru|                103.27|                89.4|
|              Brunei|                100.48|               99.49|
|            Pitcairn|                 100.0|               100.0|
|               Macao|                 97.77|               92.39|
|                Cuba|                 96.37|               89.67|
|            Portugal|                 95.62|               86.75|
|           Hong Kong|                 92.41|               90.85|
|               Chile|                 92.27|               90

### Ejercicio L: Calcular el total de casos y muertes en el ultimo mes disponible

In [17]:
query = """
SELECT date_format(date, 'yyyy-MM') AS mes,
    MIN(date) AS desde, MAX(date) AS hasta,
    SUM(new_cases) AS casos, SUM(new_deaths) AS muertes
FROM covid_data
WHERE continent IS NOT NULL
    AND date_format(date, 'yyyy-MM') = (SELECT date_format(MAX(date), 'yyyy-MM') FROM covid_data)
GROUP BY date_format(date, 'yyyy-MM')
"""


result = spark.sql(query)
result.show()

+-------+----------+----------+-----+-------+
|    mes|     desde|     hasta|casos|muertes|
+-------+----------+----------+-----+-------+
|2024-08|2024-08-01|2024-08-14|47169|    815|
+-------+----------+----------+-----+-------+



### Ejercicio M. El día con más muertes nuevas en Argentina

Devolver la fecha y `new_deaths` de ese día.


In [18]:
query = """
SELECT date, new_deaths
FROM covid_data
WHERE location = 'Argentina'
ORDER BY new_deaths DESC
LIMIT 1
"""

result = spark.sql(query)
result.show()


+----------+----------+
|      date|new_deaths|
+----------+----------+
|2021-05-23|      4516|
+----------+----------+



### Ejercicio N. Países por continente

Contar cuántos `location` distintos hay en cada `continent`. Dejar afuera las filas en las que `continent` es nulo.


In [19]:
query = """
SELECT continent, COUNT(DISTINCT location) AS paises
FROM covid_data
WHERE continent IS NOT NULL
GROUP BY continent
ORDER BY paises DESC
"""

result = spark.sql(query)
result.show()


+-------------+------+
|    continent|paises|
+-------------+------+
|       Africa|    58|
|       Europe|    55|
|         Asia|    51|
|North America|    41|
|      Oceania|    24|
|South America|    14|
+-------------+------+

